<a href="https://colab.research.google.com/github/mdi-group/royce-psdi-crystallm-training/blob/master/notebooks/Z_finetune_density_example_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mdi-group/royce-psdi-crystallm-training/blob/master/notebooks/Z_finetune_density_example_colab.ipynb)

## Finetuning CrystaLLM-$\pi$ on a custom property dataset

Hopefully by now you have some understanding on how to get a language model `speak` crystal structure language using CIFs, and how we can approach the problem of generating crystal structures with a particular functional property constraint in mind.

In this notebook, we will see how to put the theory we've learned into motion using CrystaLLM-$\pi$ repository. The steps we will take here are similar to any property you may want to fine-tune on.

That is to say, we need to find a dataset of crystal structures that we can turn into CIF Files, format them in the way our model expects them, retrieve property labels associated to each CIF, then we fine-tune our model to find patterns between the CIFs and the properties, so we can flip the problem around and task the model to generate a full CIF given our property constraint.

The repository has a large number of scripts to help us along the way so we don't need to code the processing steps from scratch everytime we want to apply our model to a different porperty conditioning task.

In [ ]:
# @title
# Colab setup for CrystaLLM-pi
# In Colab, set Runtime -> Change runtime type -> T4 GPU or better.
# Add an HF_TOKEN secret in Colab Secrets and enable notebook access.

from pathlib import Path
import os
import signal
import subprocess
import sys

IS_COLAB = Path("/content").exists()
REPO = Path("/content/CrystaLLM-pi") if IS_COLAB else Path.cwd()
DONE = Path("/content/.crystallm_pi_setup_done") if IS_COLAB else Path(".crystallm_pi_setup_done")

if IS_COLAB and not DONE.exists():
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "uv"])

    if not (REPO / ".git").exists():
        subprocess.check_call([
            "git", "clone", "--depth", "1",
            "https://github.com/C-Bone-UCL/CrystaLLM-pi.git",
            str(REPO),
        ])

    subprocess.check_call(["uv", "pip", "install", "--system", "-r", "requirements.txt"], cwd=REPO)
    subprocess.check_call([
        "uv", "pip", "install", "--system",
        "git+https://github.com/lematerial/material-hasher.git",
        "git+https://github.com/KellerJordan/Muon",
    ], cwd=REPO)
    subprocess.check_call(["uv", "pip", "install", "--system", "-e", "."], cwd=REPO)

    # The notebook only uses text/CIF models. Removing these avoids a common
    # Colab torch/torchvision binary mismatch when importing transformers.Trainer.
    subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "torchvision", "torchaudio"])

    DONE.write_text("done\n")
    print("Setup complete. Restarting the Colab runtime. After reconnecting, run from this cell again.")
    os.kill(os.getpid(), signal.SIGKILL)

if IS_COLAB:
    os.chdir(REPO)
    print(f"Working in {REPO}")


In [ ]:
import __init__

In [ ]:
# @title
# Colab secrets and downloads

from pathlib import Path
import json
import os
import subprocess
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

os.environ["PYTHONWARNINGS"] = "ignore::FutureWarning,ignore::UserWarning"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

try:
    from google.colab import userdata
except Exception:
    userdata = None

from google.colab import output
output.enable_custom_widget_manager()

def get_secret(name):
    if userdata is not None:
        try:
            value = userdata.get(name)
            if value:
                return value
        except Exception:
            pass
    return os.environ.get(name)


HF_TOKEN = get_secret("HF_TOKEN") or get_secret("HF_KEY") or get_secret("HF_HUB_TOKEN")
WANDB_KEY = get_secret("WANDB_API_KEY") or get_secret("WANDB_KEY") or "dummy"

if not HF_TOKEN:
    raise RuntimeError("Add an HF_TOKEN secret in Colab Secrets and enable notebook access.")

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HF_KEY"] = HF_TOKEN
os.environ["HF_HUB_TOKEN"] = HF_TOKEN
os.environ["WANDB_DISABLED"] = "true"
if WANDB_KEY != "dummy":
    os.environ["WANDB_API_KEY"] = WANDB_KEY

Path("API_keys.jsonc").write_text(json.dumps({"HF_key": HF_TOKEN, "wandb_key": WANDB_KEY}, indent=2))

from huggingface_hub import HfApi, login
login(token=HF_TOKEN)
HF_USERNAME = HfApi().whoami(token=HF_TOKEN)["name"]
print("Hugging Face user:", HF_USERNAME)

# Download the pretrained base checkpoint expected by data/config.jsonc.
BASE_MODEL_DIR = Path("model_ckpts/mpdb-small-base-lematerial/checkpoint-1250000")
BASE_MODEL_DIR.mkdir(parents=True, exist_ok=True)
BASE_MODEL_URL = "https://huggingface.co/c-bone/CrystaLLM-pi_base/resolve/main"
for filename in ["config.json", "model.safetensors"]:
    target = BASE_MODEL_DIR / filename
    if not target.exists() or target.stat().st_size == 0:
        subprocess.check_call(["wget", "-c", "-O", str(target), f"{BASE_MODEL_URL}/{filename}"])
    print("Base model file:", target)

# Avoid requiring a Weights & Biases key when report_to is not wandb.
train_path = Path("_train.py")
train_text = train_path.read_text()
old = "    login(token=hf_key_json)\n    wandb.login(key=wandb_key)\n"
new = """    login(token=hf_key_json)
    if args.report_to == \"wandb\":
        wandb.login(key=wandb_key)
    else:
        os.environ[\"WANDB_DISABLED\"] = \"true\"
"""
if old in train_text and "if args.report_to == \"wandb\":" not in train_text:
    train_path.write_text(train_text.replace(old, new))
    print("Patched _train.py to skip wandb.login unless report_to='wandb'.")


### 0. Getting our data

First thing we need to do is find a data that contains crystal structures. These may be stored in different formats (CIFs, `ase.Atoms` objects, (Atom ID, Atom Coordinates, Lattice Parameters) format, etc...).

Thankfully, python libraries like `Pymatgen` or `ase` allow us to switch between data types pretty easily so we can offload the format type conversions pretty easily.

In our example we are going to use a small dataset of 1,000 crystal strcutures taken from the [MP-20 dataset](https://github.com/txie-93/cdvae/blob/main/data/mp_20/README.md), you can have a look at the original paper's `README.md` linked for more information and some structure visualisations.

"MP-20 ([Xie et al., 2022](https://arxiv.org/abs/2110.06197)) contains 45,231 metastable crystal structures from the Materials Project ([Jain et al., 2013](https://pubs.aip.org/aip/apm/article/1/1/011002/119685/Commentary-The-Materials-Project-A-materials)), each with up to 20 atoms and spanning 89 different element types." (definition from https://papersgraph.com/datasets/mp20)

It's ideal for experimentation with the model because the crystal structures are quite simple, fully ordered, meta-stable and the structures have relatively small unit cells.

For the sake of this notebook I have already sampled a set of 1,000 rows from the original data and stored it in the tutorial repo.

Lets write come code to download the raw data:

In [ ]:
import pandas as pd
from pathlib import Path

# Download only the CIF and material ID columns
url = "https://raw.githubusercontent.com/mdi-group/royce-psdi-crystallm-training/master/data/mp-20-balanced-1k-toy-dataset"
df = pd.read_csv(url, usecols=["cif", "material_id"])

# CrystaLLM-pi uses DataFrames to organise materials as a table:
# each row is one material, while columns store its CIF, ID and database.
# This makes the data easy to filter and standardize
df = df.rename(columns={"cif": "CIF", "material_id": "Material ID"})
df.insert(0, "Database", "MP-20")

# Save the finished dataframe
Path("data").mkdir(exist_ok=True)
df.to_parquet("data/mp-20-1k.parquet", index=False)

> You may notice we use the `.parquet` format. We prefer this over `.csv` because it compresses well, preserves data types, can load selected columns efficiently (less memory overhead), which makes it better for large datasets we often use in our workflows. The only downside is that we cant directly click the file to look inside because its compressed, but we can still easily load and visualise in the notebook.

### 1. Making the data compatible with the CrystaLLM-pi Pipeline

The cell above shows how we would download data for most crytsal structure datasets.

Now, we want to customize our dataset a little bit so it fits in with the CrystaLLM-pi pipeline.

First, we symmetrize the CIFs. Alot of databases will come with CIFs or structures with `P 1` symmetry labels even though they can be high symmetry. We can use `Pymatgen's` symmetry analyzer to add symmetry information to the CIFs. This is important because CrystaLLM-pi leverages symmetry information to learn patterns in crystal structures during learning.

Then, we want to add the Reduced Formula column because it allows us to have quickly accessible and informative labels for the CIFs in our database, which are used in downstream processing and metrics steps for the pipeline.

Finally, we add the property of interest. Here we asdd density as it is easy to calculate, intrrpret, and visualise, but any numerical property would in theory be compatible.

In [ ]:
import numpy as np
from tqdm.auto import tqdm
from pymatgen.core import Structure
from pymatgen.io.cif import CifWriter

processed_cifs, densities, formulas = [], [], []
failed_transformations = 0

# I love tqdm
# it is a little wrapper for our loop that allows us to track progress
for cif in tqdm(df["CIF"], desc="Processing CIFs"):
    try:
        # turn CIF into structure object
        structure = Structure.from_str(cif, fmt="cif")

        # add symmetry info
        processed_cifs.append(str(CifWriter(structure, symprec=0.1)))
        # add densiy info in g/cm^3
        densities.append(structure.density)
        # Add reduced formula
        formulas.append(structure.composition.reduced_formula)

    except Exception:
        failed_transformations += 1
        processed_cifs.append(cif)
        densities.append(np.nan)
        formulas.append(None)

# Add the processed data to the DataFrame
df["CIF"] = processed_cifs
df["Density (g/cm^3)"] = densities
df["Reduced Formula"] = formulas

print(f"There were {failed_transformations} failed transformations")

# Reorder the column names (not necessary)
df = df[["Database", "Material ID", "Reduced Formula", "Density (g/cm^3)", "CIF"]]
df.to_parquet("data/mp-20-1k.parquet", index=False)

#### 2. Visualisation

Before starting our project or building any type of model, we need to have an overview of the data we are working with. We may want to ask:

- Does the text data look like we expect it to (visualise some crystal structures)
- Is the dataset well balanced? Typically in machine learning if a model only sees the same type of data, it is difficult for it to learn general patterns (plot the distirbution of density)

In [ ]:
# lets look at the first few rows

df.head()

In [ ]:
import matplotlib.pyplot as plt
from pymatgen.core import Structure
from pymatgen.io.ase import AseAtomsAdaptor
from ase.visualize.plot import plot_atoms

# Select three random structures
examples = df.sample(3, random_state=2)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

for ax, (_, row) in zip(axes, examples.iterrows()):
    structure = Structure.from_str(row["CIF"], fmt="cif")
    atoms = AseAtomsAdaptor.get_atoms(structure)

    plot_atoms(atoms, ax, radii=0.5)
    ax.set_title(f"{row['Reduced Formula']} ({row['Material ID']})")
    ax.axis("off")

plt.show()

> Note: Change the random seed, see what happens! You can have a look at https://docs.ase-lib.org/ase/visualize/visualize.html#matplotlib for more details on ase atoms plotting

In [ ]:
import matplotlib.pyplot as plt

# Check whether the dataset contains a balanced range of densities
df["Density (g/cm^3)"].hist()

plt.xlabel("Density (g/cm^3)")
plt.ylabel("Number of structures")
plt.title("Density distribution")
plt.grid(False)
plt.show()

Wow! The density distribution seems uniformly distributed. You may suspect and fairly so that this type of distribution seems a bit unnatural as we would expect a sort of bell curve which peaks around 6-8 g/cm^3.

This is because when I sampled the 1,000 structures for this example dataset, I made sure the densities were balanced in order to make sure the model sees a wide and diverse spread of properties during training. Generally, we like to prioritize diversity in datasets rather than sheer volume to promote pattern learning in our ML model.

> **Optional:** Can you think of other dataset properties you may want to visualise? What about how many atoms are the unit cells of our crystal structures, or the distribution of atom types in our dataset?
>
> This allows to think if there are there some imbalances we need to be weary of when training or evaluating our model.
>
> **Helpful documentation:** See the pymatgen documentation for [Structure.num_sites](https://pymatgen.org/pymatgen.core.html#pymatgen.core.structure.SiteCollection.num_sites), [Structure.composition.get_el_amt_dict()](https://pymatgen.org/pymatgen.core.html#pymatgen.core.composition.Composition.get_el_amt_dict). The pandas documentation for explode() and value_counts() may also help when counting elements across the dataset.

### 3. Clean and Standardize our dataset

Our dataset is now ready to go into the pipeline.

Before we train our machine learning model, we want to standardize formatting inside the CIF.

The script we call below checks that each CIF can be processed, standardises the formula and symmetry information, adds useful atomic properties, removes comments, and rounds long decimal values. Structures that cannot be cleaned successfully are removed.

The script can also normalise numerical properties. Here, we linearly normalise density so that its values are on a scale from 0 to 1 (Min-Max linear scaling). Keeping properties on a similar scale makes model training more stable (See [this blogpost](https://www.londonacademyofit.co.uk/resources/statistics-maths/feature-scaling) for more detals on feature scaling)

In [ ]:
# Clean the CIFs and linearly normalise the density values
!python _utils/_preprocessing/_cleaning.py \
    --input_parquet "data/mp-20-1k.parquet" \
    --output_parquet "data/mp-20-1k_clean.parquet" \
    --property_columns '["Density (g/cm^3)"]' \
    --property1_normaliser linear \
    --num_workers 2 \
    --filter_to 1024

What the CLI arguments do:

| Argument | What it does |
| -- | -- |
| `--input_parquet "data/mp-20-1k.parquet"`| Loads the Parquet dataset containing the CIFs to clean. |
| `--output_parquet "data/mp-20-1k_clean.parquet"` | Saves the cleaned dataset to this file. |
| `--property_columns '["Density (g/cm^3)"]'` | Selects density as the numerical property to normalise. |
| `--property1_normaliser linear` | Scales density values to a range from 0 to 1, saving them in `norm_Density (g/cm³)`. |
| `--num_workers 2` | Uses two CPU workers to process CIFs in parallel. |
| `--filter_to 1024` | Removes CIFs that exceed the model context length of 1,024 tokens. |

> A note on the `filter_to` CLI argument and context windows in LLMs:
>
> A model’s **context window** is the maximum amount of text chunks it can read at once. For CrystaLLM-$\pi$, each CIF is converted into sequence chunks, known as tokens, and the model can only process up to a chosen limit (set by typical CIF token lengths and compute power limits), here **1,024 tokens**.
>
> We filter out longer CIFs so that every training example fits inside the model without being cut off. Otherwise, part of the crystal structure could be truncated, leaving the model with incomplete or invalid structural information.
>
> Because we are using the MP-20 dataset, we notice that none of the CIFs are actually above context length. This is because as mentioned before the structures all have small unit cells. In more realistic and complicated datasets with larger cells, we may start bleeding above the context length.
>
> We can train the model with an extended context length to avoid discarding structures, but this is only really worth it if there are enough large structures to learn from.

##### Visualise what our dataset looks like now

In [ ]:
df_clean = pd.read_parquet("data/mp-20-1k_clean.parquet").head()
df_clean

> Have a look at a random CIF in the augmented dataset (eg. `print(df_clean['CIF'][0])`). What differences can you notice between the standard pymtagen format CIF and the format we use to train our model? Why do you think these changes were made?

**Optional:** If you look at the [python script we ran](https://github.com/C-Bone-UCL/CrystaLLM-pi/blob/main/_utils/_preprocessing/_cleaning.py), you may notice there is a `--count_tokens` CLI argument. Rerun the cleaning stage and you can visualise the distribution of token counts for each CIF in the dataset.

You may want to ask yourself:
- Are we close to reaching the context limit often?
- Is the atom count in the unit cell directly correlated to token counts? Or is it a bit more convoluted?

(*Hint: If two crystal structures have the same composition but a different symmetry, how may this affect the length of the CIF?*)

You could also have a look at the `norm_Density (g/cm^3)` distribution, how has scaling affected the distibution?

### 4. Upload our dataset to the Hugging Face hub

Alright! Our dataset is prepped and ready to be fed to a machine learning model. To promote open science and because its as pretty neat and easy way to store out datasets, we use the Hugging Face Hub to store our dataset. Our ML model loads datasets from huggingface directly when training.

This step allows us to have any datasets we make stored on a cloud so its easy to ship between computers, and also this way we can share datasets eassily within the community or allow others to reproduce our results.



You may have also heard of [dataset splitting in machine learning](https://blog.roboflow.com/train-test-split/). A dataset is commonly separated into:

- A **training** set (60-80 \%), which the model uses to learn patterns in the data, the model parameters are updated solely on this set.
  
- A **validation** set (10-20 \%), which contains structures the model does not train on. We use it during training to check whether the model is learning patterns that also work on unseen structures and optimize our model hyper-parameter choices.
  
- A **test** set (10-20 \%), which is normally kept completely separate until the end and used to report the model’s final performance.

This separation helps us identify overfitting. A model may become very good at remembering its training examples without learning patterns that generalise to new data. The validation set helps us notice when this starts to happen.

In this project, we will use training and validation splits, but we don't need a test split. A test set is particularly useful when a model is making direct predictions, such as using testing if the model can recover a known 'True' CIF from a given composition prompt.

In this tutorial, our goal is to generate entirely new crystal structures. We will therefore evaluate the model by examining if the materials generated by the model are valid or if they have the density that we request at inference.

In [ ]:
!python _utils/_preprocessing/_save_dataset_to_HF.py \
    --HF_username "{HF_USERNAME}" \
    --input_parquet 'data/mp-20-1k_clean.parquet' \
    --output_parquet 'data/mp-20-1k_final.parquet' \
    --valid_size 0.20 \
    --save_hub

> You can now go to your Hugging Face Profile and visualise your dataset on the Hub. This now can be shared with anyone!

What the CLI arguments do:

| Argument | What it does |
| -- | -- |
| `--HF_username "c-bone"` | Sets the Hugging Face account where the dataset will be uploaded. |
| `--input_parquet "data/mp-20-1k_clean.parquet"` | Loads the cleaned Parquet dataset. |
| `--output_parquet "data/mp-20-1k_final.parquet"` | Sets the dataset name. On the Hugging Face Hub, this becomes `c-bone/mp-20-1k_final`. |
| `--valid_size 0.20` | Places 20% of the structures in the validation set. The remaining 80% are used for training. |
| `--save_hub` | Uploads the finished training and validation splits to the Hugging Face Hub. |

Because we dont provide `--test_size`, no test split is created.


We are now ready to fine-tune our model. As you may have noticed, we spent a whole lot of time on data processing, **even** though we already have scripts to automate the main steps. That is because:

1. Understanding data is key to figuring out source of wierd results and potential training problems.
2. Data prcessing is arguably the most important part of a machine learning pipeline, even though it may not be as exciting as training a model. This is because the bulk of complexities of training can be automated in the backend of the code, but data is what varies most from project to project and is the part that requires the user to input the most of his domain knowledge.


### 5. Fine-tune a density-conditioned model

To train our model, we start from a pretrained unconditional CrystaLLM-$\pi$ checkpoint and teach it to use density as an additional generation constraint. Starting from a base model preserves its existing understanding of CIF syntax, composition, symmetry, and atomic coordinates.

For each training example:

1. The CIF is tokenized into a sequence of at most 1,024 tokens.
2. The normalized density is read from `norm_Density (g/cm^3)`.
3. The PKV condition encoder converts that value into learned key-value pairs that guide every Transformer layer.
4. The model learns to predict the next CIF token while using the requested density as context.

The pretrained Transformer and the new conditioning layers are optimized together. The base model uses a small learning rate so that its existing structural knowledge changes gradually, while the new conditioning layers use a larger learning rate because they must learn the density-to-structure relationship from scratch.

Validation loss is evaluated throughout training. Checkpoints are written to `output_dir`, training can stop early when validation loss no longer improves, and the best validation checkpoint is loaded at the end.

> Density must be normalized in the same way during training and generation. The model receives `norm_Density (g/cm^3)`, not the original density in $\mathrm{g\,cm^{-3}}$.

> The example config expects the pretrained checkpoint at `model_ckpts/mpdb-small-base-lematerial/checkpoint-1250000`. This directory must exist before training, or `pretrained_model_dir` must be changed to another compatible base checkpoint.

#### Training configuration

The run is controlled by [`data/config.jsonc`](../data/config.jsonc). The table below covers the parameters most commonly changed for a new finetuning run. See [`_args.py`](../_args.py) for every available argument and its default value.

| Parameter | What it does |
| -- | -- |
| `dataset_HF` | Selects the Hugging Face dataset containing the training and validation splits. |
| `context_length` | Sets the maximum number of CIF tokens processed per structure. It should match the limit used during cleaning. |
| `condition_columns` | Lists the normalized dataset columns used as generation conditions. Column names must match the dataset exactly. |
| `activate_conditionality` | Selects the conditional architecture. `PKV` injects the density through learned key-value pairs. |
| `n_prefix_tokens` | Controls how many ghost key-value pairs for conditioning are supplied to each Transformer layer. |
| `n_hidden_cond` | Sets the hidden width of the condition encoder. |
| `n_hidden_cond` | Sets the hidden width of the condition encoder. |
| `cond_dropout` | Share conditional key-value projections across all layers, limiting expressivity. |
| `n_embd`, `n_layer`, `n_head` | Describe the width, depth, and attention layout of the base Transformer. These must be compatible with the pretrained checkpoint. |
| `train_batch_size`, `eval_batch_size` | Set the number of structures processed per GPU during training and validation. |
| `learning_rate` | Sets the learning rate for the pretrained Transformer parameters. |
| `cond_lr` | Sets the separate learning rate for the newly added conditioning parameters. |
| `lr_scheduler_type`, `warmup_ratio` | Control how the learning rate changes over the course of training. |
| `pretrained_model_dir` | Points to the base-model checkpoint used to initialize the finetune. |
| `output_dir` | Sets where checkpoints, training state, and loss histories are saved. |
| `max_steps` | Sets the maximum number of optimizer updates. Early stopping may end training sooner. |
| `eval_steps`, `logging_steps` | Control how often validation runs and training metrics are reported. |
| `early_stopping_patience` | Stops training when validation loss fails to improve for several evaluations. |
| `load_best_model_at_end` | Makes sure the checkpoint with best validation loss is saved at end of training. |
| `fp16`, `torch_compile` | Reduce memory use and improve training speed on compatible GPUs. |

Arguments written after `--config` override the corresponding JSONC values without modifying the file.

In [ ]:
# Minimal Colab config file: same notebook settings, with Colab-safe paths and batch size.

import json
from pathlib import Path
import torch

config = {
  "dataset_HF": f"{HF_USERNAME}/mp-20-1k_final",
  "context_length": 1024,
  "condition_columns": "['norm_Density (g/cm^3)']",
  "n_prefix_tokens": 2,
  "n_hidden_cond": 256,
  "share_layers": "True",
  "cond_dropout": 0.01,
  "cond_lr": 5e-4,
  "cond_wd": 0.01,
  "activate_conditionality": "PKV",
  "n_embd": 512,
  "n_layer": 8,
  "n_head": 8,
  "residual_dropout": 0.1,
  "embedding_dropout": 0.1,
  "attention_dropout": 0.1,
  "train_batch_size": 4 if torch.cuda.is_available() else 1,
  "eval_batch_size": 4 if torch.cuda.is_available() else 1,
  "gradient_accumulation_steps": 8 if torch.cuda.is_available() else 1,
  "learning_rate": 5e-6,
  "lr_scheduler_type": "cosine_with_min_lr",
  "lr_scheduler_kwargs": {"min_lr_rate": 0.01},
  "warmup_ratio": 0.02,
  "adam_beta1": 0.9,
  "adam_beta2": 0.999,
  "grad_clip": 1.0,
  "weight_decay": 0.01,
  "output_dir": "data/MP-20-Density/",
  "save_total_limit": 2,
  "pretrained_model_dir": "model_ckpts/mpdb-small-base-lematerial/checkpoint-1250000",
  "eval_strategy": "steps",
  "eval_steps": 100,
  "logging_steps": 50,
  "save_strategy": "steps",
  "max_steps": 500,
  "early_stopping_patience": 2,
  "early_stopping_threshold": 1e-5,
  "seed": 2,
  "data_seed": 1,
  "load_best_model_at_end": True,
  "metric_for_best_model": "eval_loss",
  "torch_compile": False,
  "fp16": bool(torch.cuda.is_available()),
}

Path("data").mkdir(exist_ok=True)
Path("data/config.jsonc").write_text(json.dumps(config, indent=2))


In [ ]:
# Display the config that will be passed to training
!sed -n 'p' data/config.jsonc

In [ ]:
# device agnostic (CPU or GPU) training
!torchrun --nproc_per_node 1 _train.py --config data/config.jsonc

In [ ]:
import json
import matplotlib.pyplot as plt

# Load the saved training losses
path = "data/MP-20-Density/checkpoint-500/losses.json"
with open(path) as file:
    losses = json.load(file)

# Plot the training and validation losses
plt.plot(losses["training_steps"], losses["training_losses"], label="Training")
plt.plot(losses["validation_steps"], losses["validation_losses"], label="Validation")

plt.xlabel("Training step")
plt.ylabel("Loss")
plt.title("Training and validation losses")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### 6. Saving the Model to the Hugging-Face Hub

After trianing our model, we want to save it to the hub. This allows us once again to make it downloadable by all, store it easily, and trasferable among computers.

In [ ]:
!python _utils/_preprocessing/_save_model_to_HF.py \
    --checkpoint "data/MP-20-Density/checkpoint-500" \
    --repo "{HF_USERNAME}/MP-20-Density"

### 7. Register our new model for the generation pipeline

Amazing! We now have a trained model, its open to all. It would be nice to check how good it is though.

The codebase is set up so that theres a few models avaiable by default when using the generation pipeline, but when generating with a custom model like here, we need to provide some specifics that we can save in a json file (which contains some disctionary with useful custom model info).

The information we need to have:

1. Custom Model Path: Whats the Hugging Face path to our open model
2. Description: Up to you to giuve a short description so we can keep track of models as we make them
3. Conditions ($N$): How many conditions are we training with (in our case its 1, just density)
4. Example conditions: If others want to use our model, whats an example of a typical value to use?
5. Max: What was the maximum value in our dataset we used to train our model? so we can properly normalise inputs as expected by the model.
6. Min: When using Min-Max Normalization, this is set to 0.
7. Normalization: What normalisation function did we use when scaling our data during training?
8. model_type: Which of the conditioning classes of models did we use? This can be found in the model checkpoint folder in the `config.json` under the "architectures" key. Can be `Base`, `PKV`, `Prepend`, `Slider`, `Raw`

Everything here is trivial to obtain except for the Max property value in our dataset which we'll need to pull from the dataframe we made before. Lets do this now:

In [ ]:
import pandas as pd

# Load the training dataset
df = pd.read_parquet("data/mp-20-1k_clean.parquet")
print(f"max density: {df['Density (g/cm^3)'].max()}")

In [ ]:
import json

model_registry = {
    f"{HF_USERNAME}/MP-20-Density": {
        "description": "Conditioning model trained on toy density dataset",
        "conditions": 1,
        "example_conditions": ["3.0"],
        "max": 22.087,
        "min": 0.0,
        "normalization": "linear",
        "model_type": "PKV",
    }
}

with open("data/my_custom_models.json", "w", encoding="utf-8") as file:
    json.dump(model_registry, file, indent=2)